In [28]:
from gensim.models import Doc2Vec
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from d2v_train import review_to_wordlist

In [29]:
train = pd.read_csv("data/labeledTrainData.tsv", header=0,
                    delimiter="\t", quoting=3)

test = pd.read_csv("data/testData.tsv", header=0,
                   delimiter="\t", quoting=3)

In [30]:
model_d2v = Doc2Vec.load("models/300features_40minwords_10context_d2v")

2026-07-06 21:49:53,331 : INFO : loading Doc2Vec object from models/300features_40minwords_10context_d2v
2026-07-06 21:49:53,352 : INFO : loading dv recursively from models/300features_40minwords_10context_d2v.dv.* with mmap=None
2026-07-06 21:49:53,352 : INFO : loading vectors from models/300features_40minwords_10context_d2v.dv.vectors.npy with mmap=None
2026-07-06 21:49:53,369 : INFO : loading wv recursively from models/300features_40minwords_10context_d2v.wv.* with mmap=None
2026-07-06 21:49:53,370 : INFO : setting ignored attribute cum_table to None
2026-07-06 21:49:53,430 : INFO : Doc2Vec lifecycle event {'fname': 'models/300features_40minwords_10context_d2v', 'datetime': '2026-07-06T21:49:53.430637', 'gensim': '4.4.0', 'python': '3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'loaded'}


In [31]:
train_data_features = np.zeros((len(train), model_d2v.vector_size), dtype="float32")

for i in range(len(train)):
    train_data_features[i] = (model_d2v.dv[f'TRAIN_{i}'])

In [32]:
test_data_features = np.zeros((len(test), model_d2v.vector_size), dtype="float32")

for i in (range(len(test))):
    raw_review = test['review'][i]
    wordlist_review = review_to_wordlist(raw_review, remove_stopwords=True)
    test_data_features[i] = model_d2v.infer_vector(wordlist_review)

D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\d2v_train.py:20: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 20 of the file D:\Codding\Education\NLP\Bag of Words Meets Bags of Popcorn\d2v_train.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(raw_text)


In [33]:
forest = RandomForestClassifier(n_estimators=100)

forest.fit(train_data_features, train["sentiment"])
result = forest.predict(test_data_features)

output = pd.DataFrame(data={"id":test["id"], "sentiment":result})
output.to_csv("results/Doc2Vec.csv", index=False, quoting=3)

# _Metrics_

In [34]:
cv_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

accuracy_scores = cross_val_score(cv_model, train_data_features, train["sentiment"], cv=5, scoring="accuracy")
print(f"Кросс-валидация Accuracy: {accuracy_scores.mean() * 100:.2f}% (разброс: +/- {accuracy_scores.std() * 100:.2f}%)")

roc_auc_scores = cross_val_score(cv_model, train_data_features, train['sentiment'], cv=5, scoring='roc_auc')
print(f"Кросс-валидация ROC AUC:  {roc_auc_scores.mean() * 100:.2f}%")

Кросс-валидация Accuracy: 85.77% (разброс: +/- 0.49%)
Кросс-валидация ROC AUC:  92.71%
